# Week 2 — ML task framing: refresh opportunity ranking

This notebook frames a content decision before training a model. It uses the anonymized starter CSV and keeps the reasoning tied to observable fields.

> **Skill note:** The requested `skills/README.md`, `framing-ml-problems`, and `flyrank/flyrank-data` files are not present in this starter repository. I followed the repository's `work/README.md`, `GUIDE.md`, data-use rules, lane guide, and data dictionary instead.

## 0. Setup and evidence

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the repository or a child directory.")

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns from {DATA_PATH.relative_to(REPO_ROOT)}")
print("Lane evidence: the repository names the starter pipeline refresh queue, defines is_declining_label, and exports refresh recommendations.")
display(raw[["content_id", "content_type", "main_intent", "impressions_90d", "sessions_90d", "trend_direction"]].head(5))

Loaded 30,000 rows and 44 columns from data\raw\content_refresh_anonymized.csv
Lane evidence: the repository names the starter pipeline refresh queue, defines is_declining_label, and exports refresh recommendations.


,content_id,content_type,main_intent,impressions_90d,sessions_90d,trend_direction
0,content_304f48230142,keyword article,transactional,3803,17,down
1,content_a1fb4e703a9e,keyword article,informational,15320,9,down
2,content_9aa793d4d895,keyword article,informational,12581,11,down
3,content_331d6c4de07b,keyword article,commercial,11751,78,stable
4,content_d99b7a2d90ca,keyword article,informational,19140,145,down


## 1. My lane as an ML task

**Selected lane: Refresh / Content Opportunity Scoring.** The repository's lane guide and starter pipeline both point to a refresh review queue: identify existing pages that deserve attention, then prioritize the queue.

**Primary ML task: ranking.** Each content item receives a priority score, and the output is ordered for a finite review capacity. Ranking fits better than a hard yes/no decision because the team can act on the top 20 or 50 pages even when the boundary is uncertain. A binary decline label can be used as an initial evaluation proxy, but it is not the final product decision.

**Assumption:** this lane is selected because the repository provides `content_refresh_anonymized.csv`, a refresh baseline script, and `refresh_queue_sample.csv`. No separate lane-selection file was present.

## 2. Target or proxy

- **Ideal real-world target:** the value of reviewing or refreshing a page now, measured by a later, leakage-safe outcome such as meaningful improvement in impressions, clicks, CTR, or qualified sessions in a defined future window.
- **Does it exist here?** No. The starter file contains trailing 90-day aggregates and a current 30-day-vs-previous-30-day trend bucket, but no post-decision outcome and no confirmed refresh-effect label.
- **Initial practical proxy:** `trend_direction == "down"`, named `provisional_decline_proxy`. This is an observed current-window bucket constructed from the starter data, not verified ground truth for refresh value. It can reward pages that are declining for reasons a refresh will not fix.
- **Likely future target column:** `future_refresh_opportunity` (binary) or `future_refresh_value` (continuous). A binary version would be 0/1; a continuous version could be a validated change in organic clicks or sessions after a fixed future window.
- **Collection/validation:** log the decision date, action type, reviewer decision, and page-level pre/post outcomes. Compare against an agreed holdout or experiment where possible; otherwise label the result observational and directional.

In [2]:
# This is an observed current-window trend bucket, explicitly separate from future ground truth.
raw["provisional_decline_proxy"] = raw["trend_direction"].eq("down").astype("int8")
print("Proxy name: provisional_decline_proxy")
display(raw["provisional_decline_proxy"].value_counts().rename(index={0:"not_down",1:"down"}).to_frame("rows"))
print("trend_direction and trend_pct are not leakage-safe features for this proxy.")

Proxy name: provisional_decline_proxy


,rows
provisional_decline_proxy,
down,16262
not_down,13738


trend_direction and trend_pct are not leakage-safe features for this proxy.


## 3. Success metric

**Primary metric: Precision@50 against the provisional proxy.** It measures the fraction of the 50 highest-ranked pages that are currently marked down. This matches the starter pipeline's review-queue framing and a realistic small batch for human review.

False positives waste editorial time, so precision matters. False negatives still matter because a missed declining page may lose traffic, but the first operational constraint is making the top of the queue useful. A transparent stale-and-visible rule is the baseline.

A useful first baseline is the rule's Precision@50. “Good enough to support action” means the ranking beats that baseline on a held-out, leakage-safe future target and the top 50 contain enough evidence for a reviewer to choose refresh, rewrite, monitor, or leave unchanged. Proxy performance alone is not enough to claim causal refresh value.

## 4. Unit of analysis as a real dataframe

In [3]:
# The starter CSV is one row per content item.
refresh_df = raw.loc[raw["impressions_90d"].gt(0) & raw["content_age_days"].ge(90)].copy()
unit_columns = ["content_id","client_id","content_type","main_intent","content_age_days",
                "days_since_last_update","impressions_90d","clicks_90d","sessions_90d",
                "ctr","avg_position","engagement_rate","scroll_rate","trend_direction",
                "provisional_decline_proxy"]
unit_df = refresh_df[unit_columns].copy()
print("One row represents one content item (an anonymized page) in the refresh review queue.")
print(f"Shape: {unit_df.shape}")
print("Column names and dtypes:")
display(unit_df.dtypes.rename("dtype").to_frame())
print("Readable sample:")
display(unit_df.head(5))
print("Missing values in relevant fields:")
display(unit_df.isna().sum().rename("missing").to_frame())
print("Proxy distribution in the selected slice:")
display(unit_df["provisional_decline_proxy"].value_counts().sort_index().rename(index={0:"not_down",1:"down"}).to_frame("rows"))

One row represents one content item (an anonymized page) in the refresh review queue.
Shape: (30000, 15)
Column names and dtypes:


,dtype
content_id,str
client_id,str
content_type,str
main_intent,str
content_age_days,int64
days_since_last_update,int64
impressions_90d,int64
clicks_90d,int64
sessions_90d,int64
ctr,float64


Readable sample:


,content_id,client_id,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,trend_direction,provisional_decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,187,20,3803,29,17,0.76,10.6,5.88,4.55,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,445,25,15320,7,9,0.05,20.3,0.00,10.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,141,20,12581,11,11,0.09,36.5,0.00,28.57,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,463,22,11751,58,78,0.49,6.2,1.28,3.45,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,263,14,19140,24,145,0.13,44.0,0.00,24.29,down,1


Missing values in relevant fields:


,missing
content_id,0
client_id,0
content_type,0
main_intent,2374
content_age_days,0
days_since_last_update,0
impressions_90d,0
clicks_90d,0
sessions_90d,0
ctr,0


Proxy distribution in the selected slice:


,rows
provisional_decline_proxy,
not_down,13738
down,16262


## 5. Sketch the target column

In [4]:
target_sketch = unit_df[["content_id","trend_direction","provisional_decline_proxy"]].head(8).copy()
target_sketch = target_sketch.rename(columns={"provisional_decline_proxy":"target_proxy__provisional_only"})
print("target_proxy__provisional_only is a constructed proxy, not an observed refresh-success label.")
display(target_sketch)
print("Target type:", target_sketch["target_proxy__provisional_only"].dtype, "| expected values: 0 or 1")

target_proxy__provisional_only is a constructed proxy, not an observed refresh-success label.


,content_id,trend_direction,target_proxy__provisional_only
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0


Target type: int8 | expected values: 0 or 1


## 6. Why ML beats a fixed rule

A fixed rule such as `days_since_last_update >= 180` and `impressions_90d >= 500` is a useful baseline, but it compresses different situations into one flag. A page can be visible but poorly positioned, have strong impressions but very low CTR, be old but stable, or have weak engagement with little traffic. Content type and intent also change how those signals should be interpreted.

A ranking model could combine age, visibility, position, CTR, engagement, content type, intent, and evidence volume, then learn which combinations put the most currently declining pages near the top. It may generalize better to changing vocabulary and context than a single threshold, while still requiring leakage checks and human review. The current trend proxy is noisy, so ML should assist prioritization rather than declare that a refresh will work.

Rules should remain as guardrails: minimum evidence thresholds, policy exclusions, safety checks, and an abstain/low-confidence path.

In [5]:
# Simple deterministic baseline: stale and visible pages first, with exposure as a tie-breaker.
baseline = unit_df.copy()
baseline["stale_visible_rule_score"] = (baseline["days_since_last_update"].ge(180).astype(int) * baseline["impressions_90d"].clip(lower=0))
baseline_ranked = baseline.sort_values(["stale_visible_rule_score","impressions_90d"], ascending=False)
k = min(50, len(baseline_ranked))
baseline_precision_at_50 = baseline_ranked.head(k)["provisional_decline_proxy"].mean()
print(f"Stale-and-visible baseline Precision@{k}: {baseline_precision_at_50:.3f}")
display(baseline_ranked[["content_id","days_since_last_update","impressions_90d","trend_direction","provisional_decline_proxy","stale_visible_rule_score"]].head(5))

Stale-and-visible baseline Precision@50: 0.700


,content_id,days_since_last_update,impressions_90d,trend_direction,provisional_decline_proxy,stale_visible_rule_score
16751,content_cf56e2e2e282,194,61678,down,1,61678
16514,content_7368877ea310,194,59472,down,1,59472
7021,content_1bfaa38ff26c,194,25715,down,1,25715
21268,content_0a91db491d14,193,13299,down,1,13299
11489,content_5feee3994adb,194,7812,down,1,7812


## 7. Action supported by the output

The output is consumed by a FlyRank content strategist or reviewer as a prioritized refresh queue.

- **High priority:** review first; choose refresh/rewrite, improve title and metadata, or investigate intent mismatch.
- **Medium priority:** review when capacity allows and compare reason codes and evidence volume.
- **Low priority:** monitor or leave unchanged until more evidence arrives.

A human reviews the recommendation. If confidence is low, the system should abstain from a strong action and place the page in a monitor/manual-review bucket. The ranking is decision support, not an automatic claim that a refresh caused or will cause recovery.

## 8. Self-check

In [6]:
checks = {
    "one primary task type named": True,
    "target/proxy defined and labeled provisional": "provisional_decline_proxy" in raw.columns,
    "success metric selected": "Precision@50",
    "real dataframe shown": isinstance(unit_df, pd.DataFrame) and len(unit_df) > 0,
    "target sketched in code": "target_proxy__provisional_only" in target_sketch.columns,
    "fixed-rule baseline explained and measured": "stale_visible_rule_score" in baseline_ranked.columns,
    "content action connected to output": True,
    "assumptions/proxy clearly labeled": True,
}
assert all(value is not False for value in checks.values())
display(pd.Series(checks, name="status").to_frame())
print("No model was trained: this notebook stops at problem framing and data understanding.")

,status
one primary task type named,True
target/proxy defined and labeled provisional,True
success metric selected,Precision@50
real dataframe shown,True
target sketched in code,True
fixed-rule baseline explained and measured,True
content action connected to output,True
assumptions/proxy clearly labeled,True


No model was trained: this notebook stops at problem framing and data understanding.
